# Capstone Topic 2: Bronze & Silver Layers

> **Phase 7 → Week 16 (Capstone) → Topic 2**

Build the Bronze and Silver layers of the ShopStream pipeline with **all production best practices** from the course applied in one place.

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import os, shutil, json, logging, time
from datetime import datetime, timezone

spark = SparkSession.builder \
    .appName("ShopStream Capstone - Bronze & Silver") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

BASE = "/tmp/shopstream_capstone"
if os.path.exists(BASE): shutil.rmtree(BASE)
for path in ["bronze/orders", "silver/orders", "dead_letter/orders", "dq_metrics"]:
    os.makedirs(f"{BASE}/{path}")

RUN_DATE  = "2024-01-15"
RUN_ID    = f"run_{RUN_DATE.replace('-','')}_001"

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("shopstream")

def info(msg, **kw):
    log.info(json.dumps({"ts": datetime.now(timezone.utc).isoformat(),
                         "run_id": RUN_ID, "msg": msg, **kw}))

print(f"ShopStream Capstone — Bronze & Silver | run_id={RUN_ID}")

## Part 1: Simulate Raw Source Data

In [ ]:
# Simulate raw orders arriving from RDS/CDC — messy, as-is
raw_orders = spark.createDataFrame([
    # Normal orders
    ("ORD-001", "USR-100", "laptop",  1299.99, "shipped",   "US",   "2024-01-15 09:00:00"),
    ("ORD-002", "USR-101", "phone",    799.00, "pending",   "EU",   "2024-01-15 09:05:00"),
    ("ORD-003", "USR-102", "tablet",   499.50, "shipped",   "APAC", "2024-01-15 09:10:00"),
    ("ORD-004", "USR-100", "headphones",149.99,"cancelled", "US",   "2024-01-15 09:15:00"),
    ("ORD-005", "USR-103", "monitor",  329.00, "shipped",   "EU",   "2024-01-15 09:20:00"),
    # Bad data — pipeline must handle these
    ("ORD-006", None,      "keyboard",  89.99, "shipped",   "US",   "2024-01-15 09:25:00"),  # null user
    ("ORD-007", "USR-104", "mouse",    -10.00, "shipped",   "US",   "2024-01-15 09:30:00"),  # negative amount
    ("ORD-001", "USR-100", "laptop",  1299.99, "shipped",   "US",   "2024-01-15 09:00:00"),  # duplicate
    ("ORD-008", "USR-105", "webcam",    59.99, "UNKNOWN",   "US",   "2024-01-15 09:35:00"),  # invalid status
], ["order_id", "user_id", "product", "amount", "status", "region", "event_time"])

print(f"Raw source rows: {raw_orders.count()}")
raw_orders.show(truncate=False)

## Part 2: Bronze Layer — Ingest As-Is

In [ ]:
def ingest_bronze(raw_df: DataFrame, run_date: str, run_id: str) -> DataFrame:
    """Bronze: keep everything, add metadata. No filtering, no transforms."""
    return raw_df \
        .withColumn("_ingested_at", F.current_timestamp()) \
        .withColumn("_source",      F.lit("orders_rds_cdc")) \
        .withColumn("_run_id",      F.lit(run_id)) \
        .withColumn("date",         F.lit(run_date))

def write_bronze(bronze_df: DataFrame, path: str, run_date: str) -> int:
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    bronze_df.write.format("delta") \
        .mode("overwrite") \
        .partitionBy("date") \
        .option("delta.enableChangeDataFeed", "true") \
        .save(path)
    count = spark.read.format("delta").load(path) \
                 .filter(F.col("date") == run_date).count()
    return count

t0    = time.time()
bronze = ingest_bronze(raw_orders, RUN_DATE, RUN_ID)
count  = write_bronze(bronze, f"{BASE}/bronze/orders", RUN_DATE)
info("bronze_complete", rows=count, duration_s=round(time.time()-t0, 2))

print(f"Bronze written: {count} rows")
spark.read.format("delta").load(f"{BASE}/bronze/orders") \
     .select("order_id","user_id","amount","status","_run_id","date") \
     .show(truncate=False)

## Part 3: Silver Layer — Validate, Deduplicate, Enrich

In [ ]:
VALID_STATUSES = ["shipped", "pending", "cancelled"]

# ── Pure transformation functions (testable) ──────────────────

def split_valid_dead_letter(df: DataFrame) -> tuple:
    """Split into (valid, dead_letter). Dead letter gets _rejection_reason."""
    reason = (
        F.when(F.col("user_id").isNull(),            F.lit("|null_user_id")).otherwise(F.lit("")) +
        F.when(F.col("amount") <= 0,                 F.lit("|non_positive_amount")).otherwise(F.lit("")) +
        F.when(~F.col("status").isin(VALID_STATUSES), F.lit("|invalid_status")).otherwise(F.lit(""))
    )
    tagged      = df.withColumn("_rejection_reason", reason)
    valid       = tagged.filter(F.col("_rejection_reason") == "").drop("_rejection_reason")
    dead_letter = tagged.filter(F.col("_rejection_reason") != "") \
                        .withColumn("_rejected_at", F.current_timestamp())
    return valid, dead_letter

def deduplicate(df: DataFrame) -> DataFrame:
    """Keep the first occurrence of each order_id by event_time."""
    from pyspark.sql import Window
    w = Window.partitionBy("order_id").orderBy("event_time")
    return df.withColumn("_rn", F.row_number().over(w)) \
             .filter(F.col("_rn") == 1).drop("_rn")

def enrich(df: DataFrame) -> DataFrame:
    """Cast types, add derived columns."""
    return df \
        .withColumn("event_ts",  F.to_timestamp("event_time")) \
        .withColumn("amount",    F.col("amount").cast(DoubleType())) \
        .withColumn("tier",
            F.when(F.col("amount") >= 1000, "premium")
             .when(F.col("amount") >= 300,  "standard")
             .otherwise("basic")
        ) \
        .select("order_id","user_id","product","amount","status","region","tier","event_ts","date")


# ── Run the Silver pipeline ───────────────────────────────────
bronze_df = spark.read.format("delta").load(f"{BASE}/bronze/orders") \
                 .filter(F.col("date") == RUN_DATE)

valid_df, dead_df = split_valid_dead_letter(bronze_df)
silver_df = valid_df.pipe(deduplicate).pipe(enrich)

# Write Silver (idempotent)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
silver_df.write.format("delta").mode("overwrite").partitionBy("date").save(f"{BASE}/silver/orders")

# Write dead-letter (append — accumulate all rejected records)
dead_df.write.format("delta").mode("append").partitionBy("date").save(f"{BASE}/dead_letter/orders")

silver_count = silver_df.count()
dead_count   = dead_df.count()
info("silver_complete", valid_rows=silver_count, rejected_rows=dead_count)

print(f"Silver: {silver_count} valid rows, {dead_count} rejected")
print("\nSilver layer:")
spark.read.format("delta").load(f"{BASE}/silver/orders").show(truncate=False)
print("\nDead letter:")
spark.read.format("delta").load(f"{BASE}/dead_letter/orders") \
     .select("order_id","user_id","amount","status","_rejection_reason").show(truncate=False)

## Part 4: Silver DQ Checks

In [ ]:
def run_silver_dq(df: DataFrame, run_date: str) -> dict:
    total = df.count()
    if total == 0:
        raise ValueError("DQ FAIL: Silver has 0 rows — pipeline likely broken")

    checks = {
        "total_rows":           total,
        "null_user_id":         df.filter(F.col("user_id").isNull()).count(),
        "null_order_id":        df.filter(F.col("order_id").isNull()).count(),
        "duplicate_order_id":   total - df.select("order_id").distinct().count(),
        "negative_amount":      df.filter(F.col("amount") <= 0).count(),
        "invalid_status":       df.filter(~F.col("status").isin(VALID_STATUSES)).count(),
        "avg_amount":           round(df.agg(F.avg("amount")).collect()[0][0] or 0, 2),
    }

    violations = []
    if checks["null_user_id"]       > 0:           violations.append("null user_id")
    if checks["duplicate_order_id"] > 0:           violations.append("duplicate order_id")
    if checks["negative_amount"]    > 0:           violations.append("negative amount")
    if checks["invalid_status"]     > 0:           violations.append("invalid status")

    checks["passed"]     = len(violations) == 0
    checks["violations"] = violations
    checks["run_date"]   = run_date
    return checks

silver_read = spark.read.format("delta").load(f"{BASE}/silver/orders") \
                   .filter(F.col("date") == RUN_DATE)

dq = run_silver_dq(silver_read, RUN_DATE)
print("Silver DQ Report:")
print(json.dumps(dq, indent=2))

# Save DQ metrics to S3 (for Airflow DQ task to read)
with open(f"{BASE}/dq_metrics/silver_{RUN_DATE}.json", "w") as f:
    json.dump(dq, f, indent=2)

if not dq["passed"]:
    raise ValueError(f"DQ failed: {dq['violations']}")
else:
    info("dq_silver_passed", rows=dq["total_rows"])

## Exercises

1. Add a Silver DQ check that verifies referential integrity: every `user_id` in Silver orders must exist in a `users` Delta table. Write the check and test it with some orphan user IDs.
2. Extend the `enrich()` function to also: (a) extract the `event_date` from `event_ts` as a Date column, (b) add `is_high_value` boolean for `amount >= 500`, (c) standardize `region` to uppercase. Write unit tests for all three additions.
3. Modify `write_bronze()` to also write a `_SUCCESS` marker file after completion (like `spark.range(1).write.mode("overwrite").text(path + "/_SUCCESS")`). The downstream Silver job should check for this marker before reading.
4. The Silver job gets a batch with 40% null `user_id` (upstream bug). The current code routes them all to dead-letter. Should it fail the pipeline or continue? Write the condition and justify.
5. Run the full Bronze → Silver pipeline twice with the same `RUN_DATE`. Verify the Silver row count is identical (idempotent). Explain which parts of the code make this idempotent.